<a href="https://colab.research.google.com/github/zakateryila/Zaka-Teryila-Peter-/blob/main/Day4_prompt_classifier_Zaka_Teryila.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!nvidia-smi

Fri Sep 25 14:06:57 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip -q install -U "transformers>=4.45.0" accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 71.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 25.2 MB/s eta 0:00:00


In [3]:
import torch, json, re
from transformers import pipeline
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct" # GPU runtime
# MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct" # <-- switch to this line if you are on CPU
llm = pipeline(
"text-generation",
model=MODEL_ID,
torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
device_map="auto",
)
print("Running on:", "GPU" if torch.cuda.is_available() else "CPU")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Running on: GPU


In [4]:
def ask(system_prompt, user_message, sample=False, temperature=0.7, max_new_tokens=120):
"""Send one system prompt + one user message to the model, return the text reply."""
messages = [
{"role": "system", "content": system_prompt},
{"role": "user", "content": user_message},
]
kwargs = {"max_new_tokens": max_new_tokens, "return_full_text": False}
if sample:
kwargs.update(do_sample=True, temperature=temperature, top_p=0.9)
else:
kwargs.update(do_sample=False) # greedy = same input, same output
out = llm(messages, **kwargs)[0]["generated_text"]
if isinstance(out, list): # some versions return the full chat
out = out[-1]["content"]
return out.strip()
# smoke test
print(ask("You are a terse assistant.", "Say OK and nothing else."))

IndentationError: expected an indented block after function definition on line 1 (4272955199.py, line 2)

In [5]:
LABELS = ["billing", "technical", "account_access", "other"]
TEST_SET = [
("I was charged twice for my September subscription. Please refund the extra 12,000.", "billing"),
("The app crashes every time I try to upload a photo on Android 14.", "technical"),
("I forgot my password and the reset email never arrives.", "account_access"),
("Do you have an office in Abuja? Just curious.", "other"),
("My card was declined but the money left my account and my plan is still inactive.", "billing"),
("Video calls freeze after about two minutes on WiFi.", "technical"),
("I want to delete my account and everything you have stored about me.", "account_access"),
("Please change the invoice address to my company name before the next receipt.", "billing"),
("Login says 'too many attempts' even though this is my first try today.", "account_access"),
("Your new logo looks great, just wanted to say well done.", "other"),
]
print(len(TEST_SET), "test cases loaded")

10 test cases loaded


In [6]:
def prompt_v1(text):
return f"Classify this customer message: {text}"
for text, gold in TEST_SET[:3]:
print("GOLD:", gold)
print("RAW :", repr(ask(SYSTEM_V1, prompt_v1(text))))
print("-" * 70)
Read the raw output carefully and write your observations

IndentationError: expected an indented block after function definition on line 1 (1477115451.py, line 2)

In [7]:
def parse_label(raw):
"""Pull a valid label out of the model's reply, or report failure."""
match = re.search(r"\{.*\}", raw, re.S) # find the first {...} block
if match:
try:
data = json.loads(match.group(0))
label = str(data.get("label", "")).strip().lower()
if label in LABELS:
return label
except json.JSONDecodeError:
pass
return "UNPARSEABLE"
def evaluate(system_prompt, build_user, sample=False):
correct = parsed = 0
for text, gold in TEST_SET:
raw = ask(system_prompt, build_user(text), sample=sample)
pred = parse_label(raw)
parsed += (pred != "UNPARSEABLE")
correct += (pred == gold)
flag = "PASS" if pred == gold else "FAIL"
print(f"{flag} gold={gold:<15} pred={pred:<15} {text[:45]}...")
n = len(TEST_SET)
print(f"\nParseable: {parsed}/{n} Accuracy: {correct}/{n} = {correct/n:.0%}")
return correct / n


IndentationError: expected an indented block after function definition on line 1 (3622661000.py, line 2)

In [8]:
score_v1 = evaluate(SYSTEM_V1, prompt_v1)

NameError: name 'evaluate' is not defined

In [9]:
SYSTEM_V2 = """You are a support-ticket routing system for a mobile fintech app.
Classify each customer message into exactly one label.
Labels:
- billing: payments, charges, refunds, invoices, pricing, failed transactions
- technical: crashes, bugs, errors, freezing, anything broken in the product
- account_access: login, passwords, OTP codes, lockouts, creating or deleting an account
- other: anything else, including praise, general questions, and company information
Rules:
- If a message mentions both money and access, label it by what the customer wants FIXED.
- Never invent a label outside the four above.
- Reply with JSON only. No explanation, no markdown, no code fences.
Output format:
{"label": "billing | technical | account_access | other", "confidence": 0.0-1.0}"""
def prompt_v2(text):
return f"Message: {text}"
score_v2 = evaluate(SYSTEM_V2, prompt_v2

IndentationError: expected an indented block after function definition on line 14 (3218077857.py, line 15)

In [10]:
FEW_SHOT = """Examples:
Message: My subscription renewed but the app still shows the Free plan.
{"label": "billing", "confidence": 0.8}
Message: The camera screen is completely black on my Samsung.
{"label": "technical", "confidence": 0.95}
Message: The OTP code never arrives so I cannot sign in.
{"label": "account_access", "confidence": 0.95}
Message: Who founded this company?
{"label": "other", "confidence": 0.9}
"""
def prompt_v3(text):
return f"{FEW_SHOT}\nNow classify this one.\n\nMessage: {text}"
score_v3 = evaluate(SYSTEM_V2, prompt_v3) # same system prompt, richer user message

IndentationError: expected an indented block after function definition on line 11 (3087308812.py, line 12)

In [11]:
def consistency(system_prompt, build_user, runs=3):
stable = 0
for text, gold in TEST_SET:
preds = {parse_label(ask(system_prompt, build_user(text), sample=True))
for _ in range(runs)}
if len(preds) == 1:
stable += 1
else:
print("UNSTABLE:", preds, "|", text[:50])
print(f"\nStable on {stable}/{len(TEST_SET)} messages over {runs} runs")
return stable / len(TEST_SET)
consistency(SYSTEM_V2, prompt_v3)

IndentationError: expected an indented block after function definition on line 1 (1480245376.py, line 2)